# Demand Forecasting - Data Understanding Figure

Standalone exploratory notebook for generating a slide-ready figure that explains why forecasting is needed in the POS project. It uses the actual transaction dataset, filters voided lines, aggregates demand over time, and exports clean PNG assets.

Report alignment checked from `rapport (1).pdf`: the demand forecasting module builds daily and weekly item demand, uses chronological data, and relies on calendar, lag, rolling, holiday/Ramadan, and hierarchy features before selecting LightGBM. This figure intentionally focuses on broad temporal and meal-period demand characteristics rather than a Ramadan-only story.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "backend" / "Ai models" / "data" / "raw" / "enterprise_pos_dataset.csv"
OUT_DIR = ROOT / "notebooks_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PATH = OUT_DIR / "figure_forecasting_data_understanding.png"
FIG_HIGHRES_PATH = OUT_DIR / "figure_forecasting_data_understanding_highres.png"
NOTE_PATH = OUT_DIR / "figure_forecasting_data_understanding_notes.md"

# Load transaction-level POS lines. The source file is pipe-delimited.
df = pd.read_csv(DATA_PATH, sep="|")
df["order_date"] = pd.to_datetime(df["order_date"])
df["is_voided"] = df["is_voided"].astype(str).str.lower().eq("true")
valid = df.loc[~df["is_voided"]].copy()

# In this generated POS dataset, each valid row is one sold item line.
# Forecasting demand is therefore represented as item-line quantity over time.
valid["demand_units"] = 1
valid["hour"] = pd.to_datetime(valid["order_time"], format="%I:%M %p", errors="coerce").dt.hour
valid["weekday"] = valid["order_date"].dt.day_name()
valid["weekday_num"] = valid["order_date"].dt.weekday
valid["month"] = valid["order_date"].dt.month
valid["month_name"] = valid["order_date"].dt.month_name().str[:3]

# Panel A: daily demand with a 30-day rolling mean.
daily = valid.groupby("order_date", as_index=True)["demand_units"].sum().sort_index()
full_dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_dates, fill_value=0)
daily_roll = daily.rolling(30, min_periods=7).mean()

# Panel B: average daily demand by weekday.
weekday_daily = daily.rename("demand_units").to_frame()
weekday_daily["weekday_num"] = weekday_daily.index.weekday
weekday_daily["weekday"] = weekday_daily.index.day_name()
weekday_avg = weekday_daily.groupby(["weekday_num", "weekday"])["demand_units"].mean().reset_index().sort_values("weekday_num")

# Panel C: average daily demand by month of year.
month_daily = daily.rename("demand_units").to_frame()
month_daily["month"] = month_daily.index.month
month_daily["month_name"] = month_daily.index.month_name().str[:3]
month_avg = month_daily.groupby(["month", "month_name"])["demand_units"].mean().reset_index().sort_values("month")

# Panel D: average item-line demand by meal period.
# This matches backend/app/ai/recommendations/models/fm_context.py::_get_meal_period.
def get_meal_period(hour):
    if 5 <= hour < 11:
        return "Breakfast"
    if 11 <= hour < 16:
        return "Lunch"
    if 16 <= hour < 22:
        return "Dinner"
    return "Late night"

meal_order = ["Breakfast", "Lunch", "Dinner", "Late night"]
valid["meal_period"] = valid["hour"].apply(get_meal_period)
meal_period_avg = (
    valid.groupby("meal_period")["demand_units"]
    .sum()
    .reindex(meal_order, fill_value=0)
    / daily.shape[0]
)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D7DBE2",
    "axes.labelcolor": "#2F3542",
    "xtick.color": "#4B5563",
    "ytick.color": "#4B5563",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "legend.fontsize": 9,
})

colors = {
    "line": "#2563EB",
    "smooth": "#DC2626",
    "weekday": "#059669",
    "month": "#7C3AED",
    "hour": "#EA580C",
    "grid": "#E5E7EB",
    "text": "#111827",
    "muted": "#6B7280",
}

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.8))
fig.suptitle("Temporal Demand Patterns in Historical POS Transactions", fontsize=16, fontweight="bold", color=colors["text"], y=0.975)
fig.subplots_adjust(left=0.07, right=0.985, top=0.895, bottom=0.135, wspace=0.22, hspace=0.34)

ax = axes[0, 0]
ax.plot(daily.index, daily.values, color=colors["line"], linewidth=0.7, alpha=0.35, label="Daily demand")
ax.plot(daily_roll.index, daily_roll.values, color=colors["smooth"], linewidth=2.2, label="30-day rolling mean")
ax.set_title("A. Historical demand trend")
ax.set_ylabel("Item lines sold per day")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(axis="y", color=colors["grid"], linewidth=0.8)
ax.legend(loc="upper left", frameon=False)
ax.tick_params(axis="x", rotation=25)

ax = axes[0, 1]
ax.bar(weekday_avg["weekday"].str[:3], weekday_avg["demand_units"], color=colors["weekday"], alpha=0.86)
ax.set_title("B. Weekly pattern")
ax.set_ylabel("Average daily demand")
ax.grid(axis="y", color=colors["grid"], linewidth=0.8)

ax = axes[1, 0]
ax.plot(month_avg["month_name"], month_avg["demand_units"], marker="o", markersize=5, color=colors["month"], linewidth=2)
ax.fill_between(np.arange(len(month_avg)), month_avg["demand_units"], color=colors["month"], alpha=0.10)
ax.set_title("C. Month-of-year seasonality")
ax.set_ylabel("Average daily demand")
ax.grid(axis="y", color=colors["grid"], linewidth=0.8)

ax = axes[1, 1]
ax.bar(meal_period_avg.index, meal_period_avg.values, color=colors["hour"], alpha=0.86, width=0.62)
ax.set_title("D. Meal-period demand pattern")
ax.set_xlabel("Meal period", labelpad=8)
ax.set_ylabel("Average demand per day")
ax.grid(axis="y", color=colors["grid"], linewidth=0.8)

for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.02)

fig.text(
    0.5, 0.04,
    "Takeaway: demand varies over time with clear trend, weekly, seasonal, and meal-period effects, justifying forecasting for planning.",
    ha="center", va="bottom", fontsize=10.5, color=colors["muted"]
)

fig.savefig(FIG_PATH, dpi=180, bbox_inches="tight", facecolor="white")
fig.savefig(FIG_HIGHRES_PATH, dpi=320, bbox_inches="tight", facecolor="white")
plt.close(fig)

note = f"""# Forecasting Data Understanding Figure Notes

Takeaway: Demand varies over time with clear trend, weekly, seasonal, and meal-period effects, which justifies the forecasting task.

## Dataset used
- Source: `{DATA_PATH.relative_to(ROOT)}`
- Raw records: {len(df):,} transaction line rows
- Valid non-voided records used: {len(valid):,} transaction line rows
- Time range: {daily.index.min().date()} to {daily.index.max().date()}
- Distinct menu items: {valid['item_name'].nunique():,}
- Distinct orders: {valid['order_id'].nunique():,}

## Aggregations performed
- Panel A aggregates valid transaction lines into daily total item demand and overlays a 30-day rolling mean.
- Panel B averages daily demand by weekday to show recurring weekly behavior.
- Panel C averages daily demand by month of year to show broad seasonality without making Ramadan the centerpiece.
- Panel D derives meal periods from order hour using the deployed FM context mapping and normalizes item-line demand by the number of observed days.

## Why this figure fits the Forecasting - Data Understanding slide
- It answers why forecasting is needed: demand is visibly non-constant across time.
- It summarizes temporal structure rather than repeating dataset schema, global counts, or generator-validation evidence.
- It supports the report's forecasting data-understanding narrative: chronological demand, calendar patterns, seasonality, and restaurant meal periods matter before forecasting preparation and modeling.
"""
NOTE_PATH.write_text(note, encoding="utf-8")

summary = {
    "dataset": str(DATA_PATH),
    "raw_rows": len(df),
    "valid_rows": len(valid),
    "date_min": str(daily.index.min().date()),
    "date_max": str(daily.index.max().date()),
    "figure": str(FIG_PATH),
    "highres": str(FIG_HIGHRES_PATH),
    "notes": str(NOTE_PATH),
}
summary
